# 06-01 优化器进阶：RMSProp 与 Adam 再理解

前面我们已经学过指数加权移动平均：

$$
v_t=\beta v_{t-1}+(1-\beta)x_t
$$

现在可以重新理解优化器。

SGD 只看当前梯度，Momentum 用 EMA 记住梯度方向，RMSProp 用 EMA 记住梯度大小，Adam 则把这两个思想合在一起。

## 1. 为什么 SGD 还不够

SGD 的更新公式是：

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

其中：

$$
g_t=\nabla_{\theta}\mathcal{L}_t
$$

SGD 的问题是：所有参数共用同一个学习率 $\eta$。

但不同参数的梯度情况可能完全不同。

有的参数梯度经常很大，说明它变化很敏感；如果还用同样的大步子，可能震荡。

有的参数梯度经常很小，说明它更新很慢；如果还用同样的小步子，可能几乎不动。

所以问题变成：**能不能给不同参数自动调整不同的实际步幅？**

## 2. 自适应学习率先别想复杂

先把学习率想成“每次改参数时迈多大一步”。

SGD 的做法比较直接：所有参数都听同一个命令。

```text
大家都走 η 这么大一步
```

但神经网络里的参数不是同一种情况。

有些参数经常被数据影响，梯度经常很大。它们像是已经被反复提醒的参数，如果还让它大步走，就容易来回晃。

有些参数很少被数据影响，梯度经常很小。它们像是很少有机会被纠正的参数，如果还只让它小步走，就可能学得特别慢。

所以自适应学习率想解决的是一句很朴素的话：

```text
不要所有参数都走同样大的步子。
```

更具体一点：

```text
经常变化很大的参数：以后谨慎一点，步子小一点
很少变化或变化较小的参数：不要压得太死，可以相对多走一点
```

AdaGrad、RMSProp、Adam 都是在围绕这件事做文章。区别只是：它们用什么方式判断“这个参数过去变化大不大”。

## 3. AdaGrad 先解决了什么

AdaGrad 的名字可以拆开看：

```text
Ada = Adaptive，自适应
Grad = Gradient，梯度
```

也就是：**根据每个参数自己的梯度情况，自动调整它的实际步子。**

它的核心做法很像给每个参数单独记一本账：

```text
这个参数过去梯度大不大？
如果过去经常很大，就说明它很敏感，以后少走点。
如果过去没怎么大过，就不要把它的步子压得太小。
```

怎么记账？

假设当前第 $t$ 次更新时，某个参数的梯度是：

$$
g_t
$$

AdaGrad 不直接只看这一次的 $g_t$，它会把这个参数过去每一次梯度的平方累加起来：

$$
s_t=s_{t-1}+g_t^2
$$

这个 $s_t$ 可以先别叫“二阶矩”这种名字，初学阶段你就把它理解成：

```text
这个参数到目前为止，梯度总共“激烈”过多少次。
```

为什么要平方？

第一，梯度有正有负，平方以后都变成正数，方便统计“大小”。

第二，较大的梯度平方后会更明显。例如：

$$
2^2=4,\qquad 10^2=100
$$

所以 $s_t$ 越大，就说明这个参数历史上越“活跃”、越“敏感”。

接着 AdaGrad 更新参数：

$$
\theta_t=\theta_{t-1}-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

这个公式不要一整坨看。它其实还是 SGD：

$$
\theta_t=\theta_{t-1}-\text{步子}
$$

只不过 AdaGrad 把原来的步子：

$$
\eta g_t
$$

改成了：

$$
\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

也就是用 $\frac{1}{\sqrt{s_t}+\epsilon}$ 给当前梯度踩了一脚刹车。

如果 $s_t$ 很大，分母大，刹车重，实际更新就小。

如果 $s_t$ 不大，分母没那么大，刹车轻，实际更新就相对大。

这就是 AdaGrad 的主线：

```text
给每个参数记历史梯度账
账越大，说明过去动得越猛
以后更新它时就自动收小步子
```

## 4. AdaGrad 的问题也很直观

AdaGrad 的优点来自这本账，问题也来自这本账。

它的账本是这样加的：

$$
s_t=s_{t-1}+g_t^2
$$

注意这个式子里只有“加”，没有“忘”。

所以 $s_t$ 只会越来越大，不会自己变小。

这会带来一个很实际的问题：训练越往后，分母越可能变大。

$$
\sqrt{s_t}+\epsilon
$$

分母越大，实际步子越小：

$$
\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

你可以把它想成：AdaGrad 一开始很聪明，知道给活跃参数减速；但它太记历史了，早期的梯度会一直压着后面的更新。

于是训练后期可能出现这种情况：

```text
模型还没学到理想位置
但很多参数的实际步子已经被压得很小
训练开始变慢，甚至像走不动了一样
```

所以 AdaGrad 适合一些梯度比较稀疏的问题，因为它会照顾那些不常被更新的参数；但在深度神经网络里，如果训练时间较长，它的学习率可能衰减得太厉害。

一句话记：

```text
AdaGrad 会自动调步子，但历史梯度账只加不减，后期容易越走越慢。
```

## 5. RMSProp 为什么出现

RMSProp 可以理解成对 AdaGrad 的一个非常自然的修改。

AdaGrad 的问题是：

```text
过去所有梯度平方都一直累计，太久以前的事情也一直算数。
```

RMSProp 就说：那我们不要把所有历史都永久记住，只看“最近一段时间”的梯度大小趋势。

这就用到了前面学过的指数加权移动平均。

AdaGrad 的记账方式是：

$$
s_t=s_{t-1}+g_t^2
$$

RMSProp 改成：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

这两个式子的区别非常重要。

AdaGrad 是：

```text
旧账 + 新账，一直加
```

RMSProp 是：

```text
旧账打个折 + 新账记一点
```

其中 $\beta$ 通常接近 $1$，比如 $0.9$ 或 $0.99$。它表示：历史信息还要保留多少。

所以 RMSProp 不会让 $s_t$ 无限增大得那么厉害，因为旧梯度的影响会慢慢变淡。

一句话记：

```text
RMSProp = AdaGrad 的账本改良版：不再永久记住所有历史，而是更看重最近的梯度情况。
```

## 6. RMSProp 的公式怎么读

RMSProp 先统计最近梯度平方的大概水平：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

再更新参数：

$$
\theta_t=\theta_{t-1}-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

这两个式子可以按三句话读。

第一句：

```text
g_t 决定这一次往哪个方向改。
```

因为梯度告诉我们损失函数上升最快的方向，所以更新时要往反方向走。

第二句：

```text
s_t 记录这个参数最近的梯度大不大。
```

如果最近梯度经常大，$s_t$ 就大；如果最近梯度比较小，$s_t$ 就小。

第三句：

```text
用 sqrt(s_t) 放在分母里，是为了自动调节实际步子。
```

如果 $s_t$ 大：

$$
\sqrt{s_t}+\epsilon \text{ 变大}
$$

那么：

$$
\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

就会变小，参数更新会更谨慎。

如果 $s_t$ 小，分母没那么大，更新就不会被压得太狠。

所以 RMSProp 的公式不是凭空来的，它就是沿着这个逻辑走出来的：

```text
先看每个参数最近梯度大不大
再根据这个大小给当前更新调步子
梯度经常大：少走
梯度经常小：别压太死
```

$\epsilon$ 是一个很小的数，主要是防止分母为 $0$，让计算更稳定。

## 7. RMSProp 解决了什么

RMSProp 主要解决两个问题。

第一，它让不同参数有不同的实际学习率。

第二，它不像 AdaGrad 那样让历史梯度平方无限累积，而是更关注近期趋势。

所以 RMSProp 比 AdaGrad 更适合非平稳的深度学习训练过程。

这里的非平稳可以理解为：训练过程中，参数一直在变，损失曲面上的当前位置一直在变，早期梯度统计不应该永远主导后期训练。

## 8. Momentum 和 RMSProp 的区别

Momentum 和 RMSProp 都用到了 EMA，但记住的东西不一样。

Momentum 记住的是梯度本身：

$$
m_t=\beta m_{t-1}+(1-\beta)g_t
$$

它关心的是：最近一段时间大方向往哪里走。

RMSProp 记住的是梯度平方：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

它关心的是：最近一段时间这个参数的梯度大不大。

所以：

```text
Momentum 解决方向抖动
RMSProp 解决不同参数步幅不一样的问题
```

## 9. Adam 是怎么组合二者的

Adam 可以理解成 Momentum 和 RMSProp 的结合。

它既记录梯度的一阶矩，也记录梯度平方的二阶矩。

一阶矩：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

二阶矩：

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

$m_t$ 像 Momentum，表示方向趋势。

$v_t$ 像 RMSProp，表示梯度大小趋势。

Adam 的核心思想是：

```text
用 m_t 决定往哪里走
用 v_t 调整每个参数走多大步
```

## 10. Adam 为什么需要偏差修正

Adam 的 $m_t$ 和 $v_t$ 通常从 $0$ 开始：

$$
m_0=0,\quad v_0=0
$$

这会让前几步的移动平均偏小。

所以 Adam 会做偏差修正：

$$
\hat{m}_t=\frac{m_t}{1-\beta_1^t}
$$

$$
\hat{v}_t=\frac{v_t}{1-\beta_2^t}
$$

这和上一节指数加权移动平均里的偏差修正是同一个思想。

修正后的 Adam 更新是：

$$
\theta_t=\theta_{t-1}-\eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$

## 11. 为什么 Adam 常用默认参数

Adam 常见默认值是：

$$
\beta_1=0.9
$$

$$
\beta_2=0.999
$$

$\beta_1=0.9$ 表示一阶矩大约参考最近 $10$ 步左右的梯度方向。

因为：

$$
\frac{1}{1-0.9}=10
$$

$\beta_2=0.999$ 表示二阶矩参考更长时间的梯度平方趋势。

因为：

$$
\frac{1}{1-0.999}=1000
$$

直觉是：方向可以稍微灵活一点，但梯度大小的统计希望更稳定。

## 12. Adam 和 AdamW 的区别

AdamW 是 Adam 的一个常见改进版本。

它主要处理的是权重衰减的实现方式。

普通 Adam 中，如果把 $L_2$ 正则化直接加进损失函数，权重衰减会和 Adam 的自适应学习率混在一起。

AdamW 的思想是：把权重衰减从梯度更新里解耦出来，单独对权重做衰减。

可以粗略理解成：

```text
Adam：自适应更新和 L2 正则容易缠在一起
AdamW：自适应更新归自适应更新，权重衰减归权重衰减
```

现在很多 Transformer、预训练模型和深度学习项目更常用 AdamW。

入门阶段先记住：AdamW 是更适合配合权重衰减的 Adam 变体。

## 13. 优化器之间的逻辑关系

可以把这些优化器放在一条线里理解：

```text
SGD
-> Momentum：给梯度方向加记忆，减少抖动
-> AdaGrad：给每个参数自适应步幅
-> RMSProp：用 EMA 改进 AdaGrad，避免历史平方梯度无限累积
-> Adam：Momentum + RMSProp，并加入偏差修正
-> AdamW：Adam + 更合理的权重衰减
```

这样看，Adam 不是突然出现的复杂公式，而是一步步解决问题累积出来的结果。

## 14. 初学者怎么记

先不要把优化器当成 API 名字背。

可以这样记：

| 优化器 | 记忆方式 |
|---|---|
| SGD | 当前梯度告诉我往哪走 |
| Momentum | 不只看当前梯度，还保留过去方向 |
| AdaGrad | 梯度大的参数以后走小点 |
| RMSProp | 只看近期梯度大小，不让很久以前一直影响现在 |
| Adam | 既记方向，又调步幅 |
| AdamW | Adam，再把权重衰减处理得更干净 |

如果只是入门训练一个普通神经网络，Adam 或 AdamW 通常是比较友好的起点。

如果你想研究最终泛化表现，可以再尝试 SGD + Momentum。

## 15. 本节总结

这一节的逻辑链是：

```text
SGD 所有参数共用同一个学习率
-> 不同参数梯度大小不同，应该有不同实际步幅
-> AdaGrad 累积历史梯度平方，但分母会越来越大
-> RMSProp 用 EMA 记录近期梯度平方，避免无限累积
-> Momentum 用 EMA 记录梯度方向
-> Adam 同时使用方向 EMA 和平方梯度 EMA
-> AdamW 进一步把权重衰减解耦
```

先记住三个公式：

RMSProp：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

Adam 一阶矩：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

Adam 二阶矩：

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

到这里，优化器这条线就比较完整了。下一步可以继续进入学习率调度：为什么训练过程中学习率常常不是固定不变的。